# 🌍 AirSentinel Cameroun
## Notebook 03 — Feature Engineering
**Responsable : PEURBAR RIMBAR Firmin — ISSEA**

### Variables créées
1. Variables de lag PM2.5 (1j, 3j, 7j)
2. Moyennes glissantes (7j, 30j)
3. Indice de stagnation
4. Variables temporelles
5. Épisodes climatiques (percentile 90 — Ceccherini 2017)
6. Encodage région

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('../data/processed/dataset_tests.xlsx')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ville', 'date'])
VAR_CIBLE = 'pm2_5_moyen'
print(f'✅ Dataset chargé : {df.shape[0]:,} lignes')

✅ Dataset chargé : 49,360 lignes


## 1. Variables de lag
**Pourquoi ?** La pollution d'hier influence celle d'aujourd'hui.

In [2]:
for lag in [1, 3, 7]:
    df[f'pm2_5_lag_{lag}j'] = df.groupby('ville')[VAR_CIBLE].shift(lag)
    print(f'✅ pm2_5_lag_{lag}j créée')

✅ pm2_5_lag_1j créée
✅ pm2_5_lag_3j créée
✅ pm2_5_lag_7j créée


## 2. Moyennes glissantes
**Pourquoi ?** Captent la tendance récente de la pollution.

In [3]:
for fen in [7, 30]:
    df[f'pm2_5_moy_{fen}j'] = df.groupby('ville')[VAR_CIBLE].transform(
        lambda x: x.rolling(window=fen, min_periods=1).mean()
    )
    print(f'✅ pm2_5_moy_{fen}j créée')

✅ pm2_5_moy_7j créée
✅ pm2_5_moy_30j créée


## 3. Indice de stagnation
**Pourquoi ?** Air chaud + sec + sans vent = pollution qui s'accumule.

In [4]:
col_temp = 'temperature_2m_max'
col_vent = 'wind_speed_10m_max'
col_hum = 'precipitation_sum'

def normaliser(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx != mn else s * 0

if all(c in df.columns for c in [col_temp, col_vent, col_hum]):
    df['indice_stagnation'] = (
        0.4 * normaliser(df[col_temp]) +
        0.4 * (1 - normaliser(df[col_vent])) +
        0.2 * (1 - normaliser(df[col_hum]))
    )
    print('✅ indice_stagnation créé')
else:
    print('⚠️ Colonnes météo manquantes — vérifier les noms')

✅ indice_stagnation créé


## 4. Variables temporelles

In [5]:
df['annee']      = df['date'].dt.year
df['mois']       = df['date'].dt.month
df['jour_annee'] = df['date'].dt.dayofyear
df['saison_code'] = df['date'].dt.month.map({
    12:0, 1:0, 2:0,
    3:1,  4:1,
    5:2,  6:2,  7:2,  8:2,  9:2,  10:2,
    11:1
})
print('✅ Variables temporelles créées')

✅ Variables temporelles créées


In [6]:
df_meteo = pd.read_csv('../data/raw/dataset_meteo_cameroun.csv')
print("Colonnes :", df_meteo.columns.tolist())
print()
print("5 premières lignes :")
print(df_meteo.head(5).to_string())

Colonnes : ['id', 'time', 'weather_code', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean', 'sunrise', 'sunset', 'daylight_duration', 'sunshine_duration', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'precipitation_hours', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'city', 'region', 'latitude', 'longitude']

5 premières lignes :
   id        time  weather_code  temperature_2m_max  temperature_2m_min  temperature_2m_mean  apparent_temperature_max  apparent_temperature_min  apparent_temperature_mean           sunrise            sunset  daylight_duration  sunshine_duration  precipitation_sum  rain_sum  snowfall_sum  precipitation_hours  wind_speed_10m_max  wind_gusts_10m_max  wind_direction_10m_dominant  shortwave_radiation_sum  et0_fao_evapotranspiration   city  region  latitude  longitude
0   1 

## 5. Épisodes climatiques
**Méthode : Percentile 90 par région**
**Référence : Ceccherini et al. 2017 — Scientific Reports**

In [7]:
PERCENTILE = 0.90  # IPCC (2014) Annex II Glossary — définition événements extrêmes

# Harmattan intense — Schepanski et al. (2007) + Knippertz et al. (2008)
# dust > p90 : poussière extrême (harmattan actif)
# precipitation < p10 : sécheresse extrême (harmattan supprime les précipitations)
if 'dust_moyen' in df.columns and col_hum in df.columns:
    seuil_dust    = df.groupby('region')['dust_moyen'].transform(
        lambda x: x.quantile(PERCENTILE))
    seuil_pluie_bas = df.groupby('region')[col_hum].transform(
        lambda x: x.quantile(0.10))  # ← p10 au lieu de p25
    df['harmattan_intense'] = (
        (df['dust_moyen'] > seuil_dust) & (df[col_hum] < seuil_pluie_bas)
    ).astype(int)
    print(f'✅ harmattan_intense : {df["harmattan_intense"].sum():,} jours')
else:
    print('⚠️ harmattan_intense non créé — colonnes manquantes')

# Épisodes de feux — Barker et al. (2020) + Gordon et al. (2023)
# co > p90 en saison sèche : CO = traceur chimique standard des feux de biomasse
if 'co_moyen' in df.columns:
    seuil_co = df[df['saison_code']==0]['co_moyen'].quantile(PERCENTILE)
    df['episode_feux'] = (
        (df['co_moyen'] > seuil_co) & (df['saison_code'] == 0)
    ).astype(int)
    print(f'✅ episode_feux : {df["episode_feux"].sum():,} jours')
else:
    print('⚠️ episode_feux non créé — co_moyen manquant')

✅ harmattan_intense : 258 jours
✅ episode_feux : 1,163 jours


## 6. Encodage région

In [8]:
df = pd.concat([df, pd.get_dummies(df['region'], prefix='region')], axis=1)
print(f'✅ Régions encodées')

df.to_excel('../data/processed/dataset_enrichi.xlsx', index=False)
print(f'✅ Dataset enrichi sauvegardé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

✅ Régions encodées
✅ Dataset enrichi sauvegardé : 49,360 lignes × 61 colonnes
